In [1]:
import itertools
from typing import Optional
import numpy as np
from numpy.typing import ArrayLike



In [3]:
def _blockmatrix_coo_coords_symmetric(
    orbitals: ArrayLike,
    edge_index: ArrayLike,
    n_supercells: int = 1,
    edge_neigh_isc: Optional[ArrayLike] = None,
    symmetrize_edges: bool = False,
):
    """Returns the coo cordinates of a block matrix.

    This function assumes that:

        - All node blocks contain nonzero entries.
        - Edge blocks for edges included in `edge_index` contain
        nonzero entries.
        - All individual blocks are dense. I.e. if a block "exists",
        there are non-zero entries for all of its elements.

    The order of the coordinates returned by this function is:

        1. Node blocks.
        2. Edge blocks.
        3. Edge blocks in reverse direction (if `symmetrize_edges == True`).

    Blocks (1) and (2) are assumed to be in row-major order. Blocks (3), if
    any, are assumed to contain the data in the exact same order as (2).

    Parameters
    ----------
    orbitals:
        for each atom, the amount of orbitals it has.
    edge_index:
        shape (2, n_edges), for each edge the indices of the atoms
        that participate. If `symmetrize_edges` is `True`, this must
        ONLY contain the edges in one of the directions.
    n_supercells:
        number of supercells in the matrix.
    edge_neigh_isc:
        shape (n_edges, ), for each edge the index of the supercell
        of the interaction. If not provided, all interactions are assumed
        to be in the unit cell.
    symmetrize_edges:
        whether we should assume that the matrix contains also the edges
        that are in the opposite direction as the ones provided in
        `edge_index`.
    """
    # Initialize the arrays to store the coordinates.
    rows = []
    cols = []

    # Store index of first orbital for each atom, as well as total number of orbitals.
    first_orb = np.cumsum([0, *orbitals])
    no = first_orb[-1]

    # First, compute the coordinates for the node blocks.
    for i_at, dim in enumerate(orbitals):
        i_start = first_orb[i_at]
        i_end = i_start + dim

        block_rows, block_cols = np.mgrid[i_start:i_end, i_start:i_end].reshape(2, -1)

        print(f"Node block for atom {i_at}: rows {i_start}-{i_end}, cols {i_start}-{i_end}")
        print(f"Block rows: {block_rows}")
        print(f"Block cols: {block_cols}")

        rows.extend(block_rows)
        cols.extend(block_cols)

    # Then, the coordinates for the edge blocks.

    # Initialize lists for symmetrized edges, which we store separately
    # so that we can append all of them at the end.
    rows_symm = []
    cols_symm = []

    # Assume unit cell interactions if edge_neigh_isc is not provided
    if edge_neigh_isc is None:
        edge_neigh_isc = itertools.repeat(0)
    else:
        edge_neigh_isc = np.array(edge_neigh_isc)

    for i_edge, ((i_at, j_at), neigh_isc) in enumerate(
        zip(edge_index.T, edge_neigh_isc)
    ):
        i_start = first_orb[i_at]
        i_end = i_start + orbitals[i_at]
        j_start = first_orb[j_at]
        j_end = j_start + orbitals[j_at]

        block_rows, block_cols = np.mgrid[i_start:i_end, j_start:j_end].reshape(2, -1)
        sc_block_cols = block_cols + no * neigh_isc

        rows.extend(block_rows)
        cols.extend(sc_block_cols)

        if symmetrize_edges:
            # Columns and rows are easy to determine if the connection is in the unit
            # cell, as the opposite block is in the transposed location.
            opp_block_cols = block_rows
            opp_block_rows = block_cols

            if neigh_isc != 0:
                # For supercell connections we need to find out what is the the supercell
                # index of the neighbor in the opposite connection.
                opp_block_cols += no * (n_supercells - neigh_isc)

            rows_symm.extend(opp_block_rows)
            cols_symm.extend(opp_block_cols)

    # Add coordinates of symmetrized edges to the list of coordinates.
    rows.extend(rows_symm)
    cols.extend(cols_symm)

    return np.array(rows), np.array(cols), (no, no * n_supercells)

In [4]:
orbitals = [3, 3, 5]  # we have 3 atoms, with 3, 3, and 5 orbitals respectively
edge_index = np.array([[0, 1], [1, 2]])  # edges between atom 0 and 1, and atom 1 and 2
_blockmatrix_coo_coords_symmetric(orbitals, edge_index)

Node block for atom 0: rows 0-3, cols 0-3
Block rows: [0 0 0 1 1 1 2 2 2]
Block cols: [0 1 2 0 1 2 0 1 2]
Node block for atom 1: rows 3-6, cols 3-6
Block rows: [3 3 3 4 4 4 5 5 5]
Block cols: [3 4 5 3 4 5 3 4 5]
Node block for atom 2: rows 6-11, cols 6-11
Block rows: [ 6  6  6  6  6  7  7  7  7  7  8  8  8  8  8  9  9  9  9  9 10 10 10 10
 10]
Block cols: [ 6  7  8  9 10  6  7  8  9 10  6  7  8  9 10  6  7  8  9 10  6  7  8  9
 10]


(array([ 0,  0,  0,  1,  1,  1,  2,  2,  2,  3,  3,  3,  4,  4,  4,  5,  5,
         5,  6,  6,  6,  6,  6,  7,  7,  7,  7,  7,  8,  8,  8,  8,  8,  9,
         9,  9,  9,  9, 10, 10, 10, 10, 10,  0,  0,  0,  1,  1,  1,  2,  2,
         2,  3,  3,  3,  3,  3,  4,  4,  4,  4,  4,  5,  5,  5,  5,  5]),
 array([ 0,  1,  2,  0,  1,  2,  0,  1,  2,  3,  4,  5,  3,  4,  5,  3,  4,
         5,  6,  7,  8,  9, 10,  6,  7,  8,  9, 10,  6,  7,  8,  9, 10,  6,
         7,  8,  9, 10,  6,  7,  8,  9, 10,  3,  4,  5,  3,  4,  5,  3,  4,
         5,  6,  7,  8,  9, 10,  6,  7,  8,  9, 10,  6,  7,  8,  9, 10]),
 (11, 11))

In [5]:
def _blockmatrix_coo_coords(
    orbitals_row: ArrayLike,
    orbitals_col: ArrayLike,
    edge_index: ArrayLike,
    n_supercells: int = 1,
    edge_neigh_isc: Optional[ArrayLike] = None,
    symmetrize_edges: bool = False,
):
    """Returns the coo cordinates of a block matrix.

    This function assumes that:

        - All node blocks contain nonzero entries.
        - Edge blocks for edges included in `edge_index` contain
        nonzero entries.
        - All individual blocks are dense. I.e. if a block "exists",
        there are non-zero entries for all of its elements.

    The order of the coordinates returned by this function is:

        1. Node blocks.
        2. Edge blocks.
        3. Edge blocks in reverse direction (if `symmetrize_edges == True`).

    Blocks (1) and (2) are assumed to be in row-major order. Blocks (3), if
    any, are assumed to contain the data in the exact same order as (2).

    Parameters
    ----------
    orbitals_row / orbitals_col:
        for each atom, the amount of orbitals it has. In case of contracted basis sets,
        rows represent the original basis, and columns the contracted basis. In case of
        non-contracted basis sets, both are equal.

    edge_index:
        shape (2, n_edges), for each edge the indices of the atoms
        that participate. If `symmetrize_edges` is `True`, this must
        ONLY contain the edges in one of the directions.
    n_supercells:
        number of supercells in the matrix.
    edge_neigh_isc:
        shape (n_edges, ), for each edge the index of the supercell
        of the interaction. If not provided, all interactions are assumed
        to be in the unit cell.
    symmetrize_edges:
        whether we should assume that the matrix contains also the edges
        that are in the opposite direction as the ones provided in
        `edge_index`.
    """
    # Initialize the arrays to store the coordinates.
    rows = []
    cols = []

    # Store index of first orbital for each atom, as well as total number of orbitals.
    first_orb_row = np.cumsum([0, *orbitals_row])
    first_orb_col = np.cumsum([0, *orbitals_col])
    no_row = first_orb_row[-1]  # Total number of orbitals in the row.
    no_col = first_orb_col[-1]  # Total number of orbitals in the column.

    # First, compute the coordinates for the node blocks.
    for dim in range(len(orbitals_row)):
        i_start_row = first_orb_row[dim]
        i_end_row = i_start_row + orbitals_row[dim]
        j_start_col = first_orb_col[dim]
        j_end_col = j_start_col + orbitals_col[dim]



        block_rows, block_cols = np.mgrid[i_start_row:i_end_row, j_start_col:j_end_col].reshape(2, -1)

        print(f"Node block for atom {dim}: rows {i_start_row}-{i_end_row}, cols {j_start_col}-{j_end_col}")
        print(f"Block rows: {block_rows}")
        print(f"Block cols: {block_cols}")

        rows.extend(block_rows)
        cols.extend(block_cols)
    raise NotImplementedError("This function is not yet implemented for non-square matrices.")
    # Then, the coordinates for the edge blocks.

    # Initialize lists for symmetrized edges, which we store separately
    # so that we can append all of them at the end.
    rows_symm = []
    cols_symm = []

    # Assume unit cell interactions if edge_neigh_isc is not provided
    if edge_neigh_isc is None:
        edge_neigh_isc = itertools.repeat(0)
    else:
        edge_neigh_isc = np.array(edge_neigh_isc)

    for i_edge, ((i_at, j_at), neigh_isc) in enumerate(
        zip(edge_index.T, edge_neigh_isc)
    ):
        i_start = first_orb[i_at]
        i_end = i_start + orbitals[i_at]
        j_start = first_orb[j_at]
        j_end = j_start + orbitals[j_at]

        block_rows, block_cols = np.mgrid[i_start:i_end, j_start:j_end].reshape(2, -1)
        sc_block_cols = block_cols + no * neigh_isc

        rows.extend(block_rows)
        cols.extend(sc_block_cols)

        if symmetrize_edges:
            # Columns and rows are easy to determine if the connection is in the unit
            # cell, as the opposite block is in the transposed location.
            opp_block_cols = block_rows
            opp_block_rows = block_cols

            if neigh_isc != 0:
                # For supercell connections we need to find out what is the the supercell
                # index of the neighbor in the opposite connection.
                opp_block_cols += no * (n_supercells - neigh_isc)

            rows_symm.extend(opp_block_rows)
            cols_symm.extend(opp_block_cols)

    # Add coordinates of symmetrized edges to the list of coordinates.
    rows.extend(rows_symm)
    cols.extend(cols_symm)

    return np.array(rows), np.array(cols), (no, no * n_supercells)


In [7]:
orbitals_row = [3, 3, 5]  # we have 3 atoms, with 3, 3, and 5 orbitals respectively
orbitals_col = [7, 7, 5]  # we have 3 atoms, with 3, 3, and 5 orbitals respectively
edge_index = np.array([[0, 1], [1, 2]])  # edges between atom 0 and 1, and atom 1 and 2
_blockmatrix_coo_coords(orbitals_row, orbitals_col, edge_index)

Node block for atom 0: rows 0-3, cols 0-7
Block rows: [0 0 0 0 0 0 0 1 1 1 1 1 1 1 2 2 2 2 2 2 2]
Block cols: [0 1 2 3 4 5 6 0 1 2 3 4 5 6 0 1 2 3 4 5 6]
Node block for atom 1: rows 3-6, cols 7-14
Block rows: [3 3 3 3 3 3 3 4 4 4 4 4 4 4 5 5 5 5 5 5 5]
Block cols: [ 7  8  9 10 11 12 13  7  8  9 10 11 12 13  7  8  9 10 11 12 13]
Node block for atom 2: rows 6-11, cols 14-19
Block rows: [ 6  6  6  6  6  7  7  7  7  7  8  8  8  8  8  9  9  9  9  9 10 10 10 10
 10]
Block cols: [14 15 16 17 18 14 15 16 17 18 14 15 16 17 18 14 15 16 17 18 14 15 16 17
 18]


NotImplementedError: This function is not yet implemented for non-square matrices.